# Lesson 27: Train, validation, test, cross-validation, and data leakage


A model can perform extremely well during development but fail on new data. Ofren, the problem is not the algorithm - it is an incorrect evaluation process


## Three datasets

We normally divide the available obesrvations into three parts

| Dataset | Purpose |
| :--- | :--- |
| Training set | Fit the model's parameters |
| Validation set | Select features, hyperparameters, calibration method, and threshold |
| Test set | Estimate final perfomance on unseen data |

The test set should represent data that were not used to make modeling decisions


## Training set

The models learns from the training set

For logistic regression, it estimates coefficients such asL

$$ \beta_0, \beta_1, ..., \beta_k $$

For a neural network, it updates its weights through gradient descent

Preprocessing parameters must also be learned from the training set. For example, standartization uses:

$$ z = \frac{x - \mu_{train}}{\sigma_{train}} $$

We calculate $\mu_{train}$ and $\sigma_{train}$ only from the training data. We then apply those same values to the validation and test sets


## Validation set

The validation set can be used to:

  - compare models
  - select hyperparameters
  - select variables
  - choose a classification threshold
  - fit or select a calibration method
  - decide when to stop training

For example, we might compare:

$$ t = 0.03, \quad t = 0.05, \quad t = 0.10 $$

and choose the threshold with the lowest validation-set business cost


## Test set

The test set is used after the modelling decisions have been made

If we repeatedly examine test performance and change the model, the model, the test set gradually becomes another validation set. Its reported performance will then be overly optimistic

A useful principle is:

`Use the test set once, at the end of the modelling process`

In real projects, occasional repeated test evalution may happen, but every decision based on the test results increases the risk of overfitting to that test set


## Data leakage

Data leakage occurs when information that would not be available at prediction time influences model training or evalution

This usually produces unrealistically strong results


### Example 1: preprocessing leakage

Incorrect process:

  1. calculate the mean and standard deviation using the entire dataset
  2. standardize everything
  3. split the data

The validation and test observations influenced the preprocessing values

Correct process:

  1. split the data
  2. fit preprocessing using the training set
  3. apply it to validation and test data


### Example 2: future information

Suppose we predict fraud at tranasction time but use:

  - the result of a later investigation
  - whether the customer requested a refund
  - whether a chargeback appeared several weeks later

These variables unavailabe when the predictions must be made. Using them is **future leakage**


### Example 3: target leakage

Suppose the target is whether a transaction if fraudulent, and we include a feature named:

$$ \text{confirmed_fraud_after_review} $$

This feature directly contains information about the target. The model's performance wouuld be meaninless


### Example 4: group leakage

One customer may have many transactions. If their appear in both trainig and test, data the model partly memorize that customer

When we want performance on unseen customers, all transactions belonging to the same customer should remain in the same dataset

Poosible grouping variables include:

  - customer ID
  - card ID
  - account ID
  - device ID


## Random splint versus time split

A random splot assumes thath observations are exchangeable and thath future data will resemble pas data

For fraud detection, behaviour changes over time:

  - criminals develop new strategies
  - customer behaviour changes
  - new products are introduced
  - fraud rates and investigation processes change

Therefore, a temporial split is often more realistic:

| Dataset | Period |
| :--- | :--- |
| Training  | January-August |
| Validation | September-Octember |
| Test | November-December |

This simulates the real situation:

$$ \text{train on the past} \rightarrow \text{predict the future} $$

With a temporial split, we must not randomly move future transacions into the training set


## Cross-validation

When the dataset is relatively small, one validation set may produce an unstable estimate

In *k*-fold corss-validation:

  1. divide the training data into *k* parts
  2. train on *k* - 1 parts
  3. validate on the remaining part
  4. repeat until evety part has been used for validation
  5. average the results

For five-fold cross-validation, every observation is:

  - used for training four times
  - used for vaidation once

The final score is:

$$ \bar{M} = \frac{M_1 + M_2 + ... + M_5}{5} $$


## Stratified cross-validation

In fraud detection, the positive class may be very rare

Stratified cross-validation attempts to preserve approximately the same fraud prevalence in every fold

For example, if the overall prevalence if 2%, each fold should also contain approximately 2% fraud

Without stratification, one fold might contain very few or even no positive observations


## Coress-validation with temporal data

Ordinary shuffled cross-validation mey create future leakage

A better approach is forward validation:

| Fold | Training period | Validation |
| :--- | :--- | :--- |
| 1 | January-March | April |
| 2 | January-April | May |
| 3 | January-May | June |

The training period occures before the validation period


## A reliable evaluation pipeline

A simlified is:

  1. separate the final test set
  2. fit preprocessing on training data
  3. select the model using validation or cross-validation
  4. select the threshold using validations data
  5. fit calibration without using the final test set
  6. evaluate once on the test set
  7. report both statistical an business metrics


## Practice

### Task 1: Identify leakage

For each situation, determine whether data leakage occurs and explain why:

  1. Standartization is fitted using all observations before the train-test split

  - its data leakage: model can find dependencies in standartized data and this can artificially inflate metrics on test dataset, but when the model will use on real data, metrics can be sufficiently lower

  2. Standartization is fitted on training data and then applied to test data

  - there is no data leakage

  3. A fraud model uses the transaction amount available at payment time

  - there is no data leakage

  4. A fraud model uses the result of an investigation completed two weeks later

  - its data leakage: model can't use data from future

  5. Transactions from the same customer appear in both training and test sets

  - its data leakage: if model knows that its the same customer, it can find a non-existent dependency between user ids

  6. The threshold is selected according to the lowest test-set business cost

  - its data leakage: we can't use test for threshold calibration

  7. The threshold is selected according to validation-set business cost and evaluated once on the test set

  - there is no data leakage



### Task 2

A model produces these ROC-AUC values in five folds:

$$ 0.76, \ 0.78, \ 0.80, \ 0.82, \ 0.84 $$

Calculate:

  1. the mean ROC-AUC
  2. the range
  3. the sample variance
  4. the sample standard deviation

Use:

$$ s^2 = \frac{\sum_{i=1}^n(x_i - \bar{x})^2}{n - 1} $$

Explain what the variation between folds tells us


**Solution:**

$$ ROC\text{-}AUC_{mean} = \frac{0.76 + 0.78 + 0.80 + 0.82 + 0.84}{5} = 0.8 $$

$$ range = 0.84 - 0.76 = 0.08 $$

$$ s^2 = \frac{(0.76 - 0.8)^2 + (0.78 - 0.8)^2 + (0.80 - 0.8)^2  + (0.82 - 0.8)^2 + (0.84 - 0.8)^2}{4} = 0.001 $$

$$ s = \sqrt{0.001} \approx 0.0316 $$


The variation between folds tell us that there is different data in folds, that means that model training on different data and is is the reason why there is volatility in ROC-AUC


### Task 3

A dataset contains transactions from January to December

You choose:

  - January-August: training
  - September-Octember: validation
  - November-December: testing

Answer:

  1. Why may this be more realistic than a random split?

  - A random split can place future observations in the training set and earlier observations in the test set. This can produce unrealistic performance, especially when fraud patterns change over time.

  2. Which set should be used to fit preprocessing?

  - for fit preprocessing training set should be used

  3. Which set should be used to choose the classification threshold?

  - for threshold selection validation set should be used

  4. Which set should be used fot the final reported result?

  - for final result testing set should be used

  5. Can December observations be used to predict or preprocess January observations?

  - No, it can't, on the contrary January observations can be used to predict December observations



### Task 4

Suppose a dataset contains 100,000 transactions from 8,000 customers

  1. Why might a random transaction-level split produce optimistic results?

  - Because transactions from the same customer may appear in both training and test data. The model is then partly evaluated on customers it has already observed.

  2. How should the data be split if we want to evaluate performance on completely new customers?

  - If we want to evaluate performance on completely new customers, we should use teporary split

  3. If we instead want to predict future transaction for existing customers, would customer-level separation always be necessary?

  - its not necessary

  4. What other grouping variables might create leakage in an anti-fraud model?

  - maybe grouping by categorical features or grouping users into fraud rings or behavioral clusters


### Task 5

1. Why should the test tset not be used to select hyperparameters?

  - Because test set is independent data which can be used only with final trained model

2. What is the difference between a validation set and a test set?

  - Validation set we use for hyperparameters selecting and in some cases for calibration, test set is only for final model testing

3. Why is stratified cross-validation useful for fraud detec|tion?

  - Because fraud data is strong imbalanced. There is a possibility that most of positive fraud will end up only in certain sets, but not all

4. Why can ordinary shuffled cross-validation be unsuitable for temporal data?

  - shuffled cross-validation is problematic because it can train on future observations and validate on earlier ones. This creates future leakage and does not simulate deployment

5. Can a model with excellent test metrics stil fail after deployment? Give two possible reasons?

  - Yes, it can. First and most likely reason is data leakage. Second reason is a problem data, preprocessing etc